In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType
from pyspark.sql.functions import from_json, col, from_unixtime, to_timestamp

## Read from bronze

In [0]:
df_bronze_trades = spark.readStream.table("workspace.bronze.trades")

## Transform data

In [0]:
trades_schema = StructType([
    StructField("type", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("qty", DoubleType(), True),
    StructField("is_buyer_maker", BooleanType(), True)
])

df_silver_trades = df_bronze_trades \
    .withColumn("parsed_data", from_json(col("raw_payload"), trades_schema)) \
    .select("parsed_data.*", "ingested_at") \
    .withColumn("trade_timestamp", col("ingested_at")) \
    .drop("type") \
    .dropna()

## Write to 

In [0]:
silver_trades_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/03_silver/trades"

query_silver_trades = (df_silver_trades.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_trades_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.silver.trades")
)
query_silver_trades.awaitTermination()
print("Hoàn tất chuyển đổi dữ liệu Khớp lệnh sang Tầng Silver: workspace.silver.trades")